# Bronze Layer: Raw Data Ingestion
## Purpose
This notebook ingests raw solar sales and operations data into the Bronze layer of our Databricks Lakehouse.
The Bronze layer enables traceability by preserving our untransformed source data.

## Source Tables
+ `solar_leads`
+ `solar_sales`
+ `solar_installations`
## Bronze Tables
+ `bronze_solar_leads`
+ `bronze_solar_sales`
+ `bronze_solar_installations`
## Design
The Bronze layer is the first stage of our medallion architecture. It includes the raw version of source data and transformations are minimal to preserve our data for processing downstream.

In [0]:
%python
spark.table("bronze_solar_leads").printSchema()
spark.table("bronze_solar_sales").printSchema()
spark.table("bronze_solar_installations").printSchema()

In [0]:
%python
from pyspark.sql import functions as F

# Read Bronze data
leads_bronze = spark.table("bronze_solar_leads")

# Clean leads table
leads_silver = (
    leads_bronze
    .withColumn("lead_id", F.trim(F.col("lead_id")))
    .withColumn("territory", F.initcap(F.trim(F.col("territory"))))
    .withColumn("lead_source", F.initcap(F.trim(F.col("lead_source"))))
    .withColumn("homeowner_type", F.initcap(F.trim(F.col("homeowner_type"))))
    .withColumn(
        "monthly_electric_bill",
        F.col("monthly_electric_bill").cast("decimal(10,2)")
    )

    .select(
        "lead_id",
        "territory",
        "lead_source",
        "homeowner_type",
        "monthly_electric_bill",
        "appt_date",
        "source_system",
        "ingested_at"
    )
)

# Silver table
leads_silver.write.format("delta").mode("overwrite").saveAsTable("silver_solar_leads")

print(f"Silver leads created: {leads_silver.count()} rows")